In [1]:
%pip install numpy
%pip install faiss-cpu
%pip install sentence-transformers
%pip install openai
%pip install python-dotenv
%pip install tqdm          
%pip install pandas       



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note

In [2]:
import numpy as np
import faiss

In [3]:
import os
import json

def load_json_files(folder_path):
    data = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".json"):
            with open(os.path.join(folder_path, filename), "r", encoding="utf-8") as f:
                try:
                    data.append(json.load(f))
                except Exception as e:
                    print(f"Error parsing {filename}: {e}")
    return data


In [4]:
records = load_json_files("./../../example_docs/lit2vec_1K_sample_records")

abstract_docs = []
for rec in records:
    emb = rec.get("abstract_embedding")
    
    # Check if embedding is a non-empty list of numbers
    if not isinstance(emb, list) or len(emb) == 0 or not all(isinstance(x, (float, int)) for x in emb):
        continue  # skip invalid embeddings

    try:
        emb_array = np.array(emb, dtype=np.float32)
        if emb_array.ndim != 1:
            continue
    except Exception as e:
        print(f"Invalid embedding format in corpus_id {rec.get('corpus_id')}: {e}")
        continue

    md = rec.get("metadata", {})
    abstract_docs.append({
        "corpus_id": rec.get("corpus_id"),
        "title": md.get("title"),
        "year": md.get("year"),
        "doi": (md.get("externalids", {}) or {}).get("DOI"),
        "authors": md.get("authors") or [],
        "venue": md.get("venue"),
        "subfield_preds": rec.get("predicted_subfield", {}),
        "isopenaccess": md.get("isopenaccess"),
        "tldr": rec.get("tldr"),
        "embedding": emb_array,
    })


if not abstract_docs:
    raise ValueError("No abstract embeddings found in records.")

# Prepare matrix and FAISS index (cosine similarity via inner product + L2 normalization)
dim_abs = abstract_docs[0]["embedding"].shape[0]
abs_matrix = np.vstack([d["embedding"] for d in abstract_docs]).astype("float32")
faiss.normalize_L2(abs_matrix)  # in-place normalization

abs_index = faiss.IndexFlatIP(dim_abs)  # inner product == cosine after normalization
abs_index.add(abs_matrix)

# Lookup helpers
idx2meta_abs = abstract_docs
corpus2idx_abs = {d["corpus_id"]: i for i, d in enumerate(abstract_docs)}


In [5]:
# Recommendation utilities

def _format_author_list(authors):
    if not authors: return "Unknown Authors"
    # authors: [{"authorId": "...", "name": "..."}, ...]
    names = [a.get("name") for a in authors if a.get("name")]
    return ", ".join(names) if names else "Unknown Authors"

def _apply_filters(candidates, *, year_range=None, min_subfield=None, open_access=None):
    """
    candidates: list of (idx, score)
    year_range: (min_year, max_year) inclusive
    min_subfield: e.g., ("Biochemistry", 0.6) => keep if predicted_subfield["Biochemistry"] >= 0.6
    open_access: True/False to filter by isopenaccess
    """
    out = []
    for i, s in candidates:
        m = idx2meta_abs[i]
        if year_range is not None:
            y = m.get("year")
            if not isinstance(y, int) or not (year_range[0] <= y <= year_range[1]):
                continue
        if min_subfield is not None:
            label, thresh = min_subfield
            conf = (m.get("subfield_preds") or {}).get(label)
            if conf is None or conf < thresh:
                continue
        if open_access is not None:
            if m.get("isopenaccess") is not open_access:
                continue
        out.append((i, s))
    return out

def _mmr_rerank(query_vec, candidates, lambda_mult=0.5, k=10):
    """
    Maximal Marginal Relevance to diversify results.
    query_vec: (dim,)
    candidates: list of (idx, score) where score is cosine sim
    """
    if len(candidates) <= k:
        return candidates
    selected = []
    candidate_set = [i for i, _ in candidates]
    cand_vecs = abs_matrix[candidate_set]  # already normalized
    q = query_vec.reshape(1, -1)
    # Precompute candidate-candidate sims
    cc = cand_vecs @ cand_vecs.T  # cosine sims
    # query-candidate sims (we already have but recompute vectorized)
    qc = (q @ cand_vecs.T).ravel()

    selected_idxs = []
    remaining = list(range(len(candidate_set)))

    for _ in range(k):
        if not remaining: break
        if not selected_idxs:
            # pick best by query similarity first
            best = max(remaining, key=lambda ridx: qc[ridx])
            selected_idxs.append(best)
            remaining.remove(best)
            continue
        # compute MMR score for each remaining
        mmr_scores = []
        for ridx in remaining:
            sim_to_query = qc[ridx]
            sim_to_selected = max(cc[ridx, sidx] for sidx in selected_idxs)
            score = lambda_mult * sim_to_query - (1 - lambda_mult) * sim_to_selected
            mmr_scores.append((ridx, score))
        best = max(mmr_scores, key=lambda x: x[1])[0]
        selected_idxs.append(best)
        remaining.remove(best)

    final = [(candidate_set[ridx], float(qc[ridx])) for ridx in selected_idxs]
    return final

def recommend_by_corpus(
    corpus_id, top_k=10, year_range=None, min_subfield=None, open_access=None, use_mmr=True
):
    """
    Given a seed paper (corpus_id), return top_k similar papers by abstract embedding.
    Filters are optional.
    """
    if corpus_id not in corpus2idx_abs:
        raise ValueError(f"corpus_id {corpus_id} not found in abstract index.")
    seed_idx = corpus2idx_abs[corpus_id]
    seed_vec = abs_matrix[seed_idx].reshape(1, -1)  # already normalized
    sims, ids = abs_index.search(seed_vec, top_k + 50)  # oversample then filter
    candidates = [(int(i), float(s)) for i, s in zip(ids[0], sims[0]) if int(i) != seed_idx]

    # Optional filters
    candidates = _apply_filters(candidates, year_range=year_range, min_subfield=min_subfield, open_access=open_access)

    # Optional MMR for diversity
    if use_mmr and candidates:
        candidates = _mmr_rerank(seed_vec.ravel(), candidates, lambda_mult=0.6, k=top_k)
    else:
        candidates = candidates[:top_k]

    return _render_recommendations(corpus_id, candidates)

from sentence_transformers import SentenceTransformer
retriever_model_abs = SentenceTransformer("intfloat/e5-large-v2")

def recommend_by_text(text_query, top_k=10, year_range=None, min_subfield=None, open_access=None, use_mmr=True):
    qvec = retriever_model_abs.encode(f"query: {text_query}", normalize_embeddings=True).astype("float32")
    if qvec.shape[-1] != abs_index.d:
        raise ValueError(f"Query vec dim {qvec.shape[-1]} != index dim {abs_index.d}. Check model.")

    sims, ids = abs_index.search(qvec.reshape(1, -1), top_k + 50)
    candidates = [(int(i), float(s)) for i, s in zip(ids[0], sims[0])]

    candidates = _apply_filters(candidates, year_range=year_range, min_subfield=min_subfield, open_access=open_access)
    if not candidates:
        return {"seed_corpus_id": None, "results": []}

    if use_mmr:
        candidates = _mmr_rerank(qvec, candidates, lambda_mult=0.6, k=top_k)
    else:
        candidates = candidates[:top_k]

    return _render_recommendations(None, candidates)


def _render_recommendations(seed_corpus_id, candidates):
    results = []
    for i, score in candidates:
        m = idx2meta_abs[i]
        results.append({
            "corpus_id": m["corpus_id"],
            "title": m["title"],
            "year": m["year"],
            "doi": m["doi"],
            "venue": m["venue"],
            "authors": _format_author_list(m["authors"]),
            "tldr": m.get("tldr"),
            "cosine_sim": round(score, 4),
        })
    return {
        "seed_corpus_id": seed_corpus_id,
        "results": results
    }



/home/lit2vec/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:

# %%
# EXAMPLES

# 1) Recommendations similar to a known paper:
seed_id = abstract_docs[0]["corpus_id"]  # pick one from your dataset
rec1 = recommend_by_corpus(
    seed_id,
    top_k=5,
    # Optional filters:
    # year_range=(2015, 2025),
    # min_subfield=("Biochemistry", 0.6),
    # open_access=True,
    use_mmr=True
)
rec1


{'seed_corpus_id': 4518629,
 'results': [{'corpus_id': 37296402,
   'title': 'Silicone Substrate with Collagen and Carbon Nanotubes Exposed to Pulsed Current for MSC Osteodifferentiation',
   'year': 2017,
   'doi': '10.1155/2017/3684812',
   'venue': 'International Journal of Biomaterials',
   'authors': 'Daniyal Jamal, Roche C. de Guzman',
   'tldr': 'Pulsatile electrical current applied to human adipose tissue-derived mesenchymal stem cells (mscs) on silicone composites enhanced osteodifferentiation and calcium deposition, with 10% cnts showing triple calcium deposition under no external current, suggesting potential for enhanced bone regeneration.',
   'cosine_sim': 0.8626},
  {'corpus_id': 11662433,
   'title': 'Delivery of Bioactive Lipids from Composite Microgel-Microsphere Injectable Scaffolds Enhances Stem Cell Recruitment and Skeletal Repair',
   'year': 2014,
   'doi': '10.1371/journal.pone.0101276',
   'venue': 'PLoS ONE',
   'authors': 'Anusuya Das, Daniel A. Barker, Tiffa

In [7]:
# 2) Recommendations from a free-text idea:
rec2 = recommend_by_text(
    "green tea polyphenols mitigating oxidative stress in neurodegeneration",
    top_k=5,
    use_mmr=True
)
rec2


{'seed_corpus_id': None,
 'results': [{'corpus_id': 209498469,
   'title': 'The Bioprotective Effects of Polyphenols on Metabolic Syndrome against Oxidative Stress: Evidences and Perspectives',
   'year': 2019,
   'doi': '10.1155/2019/6713194',
   'venue': 'Oxidative Medicine and Cellular Longevity',
   'authors': 'Kui Liu, Miao Luo, Shuang Wei',
   'tldr': 'Polyphenols, including flavonoids and nonflavonoids, exhibit significant health benefits by increasing antioxidant capacity and oxygen scavenging capacity, alleviating metabolic syndrome symptoms such as central obesity and dyslipidemia by balancing the organic oxidoreductase enzyme system and restoring mitochondrial function.',
   'cosine_sim': 0.8336},
  {'corpus_id': 235286348,
   'title': 'The chemical changes in the total antioxidant status and biological activity of GSTP1 polymorphism on nasopharyngeal carcinoma patients',
   'year': 2021,
   'doi': '10.1088/1755-1315/713/1/012049',
   'venue': '',
   'authors': 'F. Farhat, M